# Baseline Classification Models

This notebook documents the classical baselines for binary Pneumonia classification. The image-level task distinguishes **Pneumonia** (`Lung Opacity`, label 1) from **Healthy** (`Normal`, label 0); `No Lung Opacity / Not Normal` images were excluded during metadata generation.

The local labeled cohort contains 14,863 images, not the paper's reported 17,489 after exclusions. Results below are measured on this project's fixed split and should not be interpreted as an exact cohort-matched reproduction.

## Dataset and fixed split

Both baselines use the single saved seed-42 stratified assignment in `metadata/split_metadata.csv`. `image_id` is the RSNA patient ID; the assignment has no cross-split ID overlap.


In [ ]:
import csv
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "metadata").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "metadata").is_dir():
    raise FileNotFoundError("Could not locate the project root")
split_path = ROOT / "metadata" / "split_metadata.csv"
with split_path.open(newline="", encoding="utf-8-sig") as f:
    split_rows = list(csv.DictReader(f))

split_counts = {}
for split in ("train", "validation", "test"):
    rows = [row for row in split_rows if row["split"] == split]
    split_counts[split] = {
        "total": len(rows),
        "pneumonia": sum(row["label"] == "1" for row in rows),
        "healthy": sum(row["label"] == "0" for row in rows),
    }

display(Markdown("| Split | Images | Pneumonia | Healthy | Share |\n|---|---:|---:|---:|---:|\n" + "\n".join(
    f"| {name} | {stats['total']:,} | {stats['pneumonia']:,} | {stats['healthy']:,} | {100 * stats['total'] / len(split_rows):.2f}% |"
    for name, stats in split_counts.items()
)))

| Split | Images | Pneumonia | Healthy | Share |
|---|---:|---:|---:|---:|
| train | 10,405 | 4,209 | 6,196 | 70.01% |
| validation | 2,972 | 1,202 | 1,770 | 20.00% |
| test | 1,486 | 601 | 885 | 10.00% |

: 

## Shared preprocessing and representation

Images are decoded dynamically from the original DICOM files, resized to 128 × 128 grayscale pixels, scaled to [0, 1], and normalized using global mean and standard deviation fitted on the training split only. Classical models consume the normalized image flattened to 16,384 features. Both model scripts use the same `PneumoniaDataset` and saved normalization statistics.

In [ ]:
preprocessing = json.loads((ROOT / "metadata" / "preprocessing_stats.json").read_text(encoding="utf-8"))
print(f"Resize: {preprocessing['image_size'][0]} × {preprocessing['image_size'][1]}")
print(f"Train-only pixel mean: {preprocessing['mean']:.8f}")
print(f"Train-only pixel standard deviation: {preprocessing['std']:.8f}")
print(f"Statistics fitted on: {preprocessing['fit_image_count']:,} training images")

## Logistic Regression

Logistic Regression was completed previously. This section loads its recorded result and does **not** retrain it. The reference values supplied for this baseline are 75.86% train accuracy and 73.02% test accuracy.

In [ ]:
logistic_result = json.loads((ROOT / "results" / "logistic_regression.json").read_text(encoding="utf-8"))
logistic_metrics = logistic_result["metrics"]
logistic_reference = logistic_result["reference_result_percent"]
print(f"Reference — train: {logistic_reference['train']:.2f}%, test: {logistic_reference['test']:.2f}%")
print(f"Measured — train: {logistic_metrics['train_accuracy_percent']:.2f}%, validation: {logistic_metrics['validation_accuracy_percent']:.2f}%, test: {logistic_metrics['test_accuracy_percent']:.2f}%")
print(f"Configuration and split SHA-256: {ROOT / 'results' / 'logistic_regression.json'}")

## SVM

The supplied paper summary does not specify the SVM kernel or hyperparameters. This experiment uses the existing **linear, squared-hinge SVM with C = 1.0**; the configuration is recorded in `results/svm.json`. The reusable implementation fits only the training split, selects an iteration using validation accuracy, and evaluates the held-out test split afterward. Running the next cell trains/evaluates SVM and refreshes its result artifact and comparison CSV; it does not run Logistic Regression or any other model.

In [ ]:
from scripts.train_svm import run as run_svm
from src.models.svm import SVMConfig

svm_result = run_svm(SVMConfig())
svm_metrics = svm_result["metrics"]
print(f"SVM — train: {svm_metrics['train_accuracy_percent']:.2f}%, validation: {svm_metrics['validation_accuracy_percent']:.2f}%, test: {svm_metrics['test_accuracy_percent']:.2f}%")

## Baseline comparison

Difference is measured test accuracy minus the paper's reference test accuracy, in percentage points. Values below are loaded from the recorded run results rather than hard-coded.

In [ ]:
with (ROOT / "results" / "classification_results.csv").open(newline="", encoding="utf-8-sig") as f:
    comparison_rows = list(csv.DictReader(f))

table = "| Model | Reference Test Accuracy | Our Test Accuracy | Difference |\n|---|---:|---:|---:|\n"
for row in comparison_rows:
    reference = float(row["reference_test_accuracy"])
    actual = float(row["test_accuracy"])
    difference = float(row["difference_from_reference"])
    table += f"| {row['model']} | {reference:.2%} | {actual:.2%} | {difference:+.2%} ({difference * 100:+.2f} pp) |\n"
display(Markdown(table))

## Discussion

These are measured results for the local Stage 2 cohort, fixed split, flattened normalized image representation, and documented configurations. The SVM result is not an exact reproduction of the reported reference: the local usable cohort has 14,863 images rather than the paper's 17,489, and the provided paper summary does not identify the SVM kernel or settings. The difference column is therefore descriptive, not evidence of a like-for-like performance gain. Logistic Regression was loaded from its existing result artifact; only the SVM cell performs model fitting.